# Exploratory Data Analysis
## Tabular-to-Image Seminar — Dataset Profiling

For each of the 3 datasets we generate:
1. Class distribution (bar chart)
2. Feature distributions (histograms)
3. Correlation heatmap
4. Missing values report
5. Summary statistics table

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.preprocessing import preprocess_dataset

sns.set_theme(style='whitegrid', font_scale=1.1)
FIG_DIR = Path('..') / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete')

---
## 1. Breast Cancer Wisconsin

In [ ]:
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
df_bc = pd.DataFrame(bc.data, columns=bc.feature_names)
df_bc['target'] = bc.target
df_bc['class'] = df_bc['target'].map({0: 'malignant', 1: 'benign'})

print(f'Shape: {df_bc.shape}')
print(f'Classes: {df_bc["class"].value_counts().to_dict()}')
print(f'Missing values: {df_bc.isnull().sum().sum()}')
df_bc.describe().round(3)

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(5, 3))
df_bc['class'].value_counts().plot.bar(ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_title('Breast Cancer — Class Distribution')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_breast_cancer_classes.png', dpi=150)
plt.show()

In [ ]:
# Feature distributions (first 10 features)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, col in enumerate(bc.feature_names[:10]):
    ax = axes[i // 5, i % 5]
    for cls, color in zip(['malignant', 'benign'], ['#e74c3c', '#2ecc71']):
        ax.hist(df_bc[df_bc['class'] == cls][col], bins=25, alpha=0.6, color=color, label=cls)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)
fig.suptitle('Breast Cancer — Feature Distributions (first 10)', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_breast_cancer_features.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 10))
corr = df_bc[bc.feature_names].corr()
sns.heatmap(corr, cmap='RdBu_r', center=0, ax=ax, 
            square=True, linewidths=0.5, 
            xticklabels=True, yticklabels=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Breast Cancer — Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_breast_cancer_corr.png', dpi=150)
plt.show()

---
## 2. Dry Bean

In [ ]:
df_db = pd.read_csv('../data/dry_bean/Dry_Bean_Dataset.csv')

print(f'Shape: {df_db.shape}')
print(f'Classes: {df_db["Class"].value_counts().to_dict()}')
print(f'Missing values: {df_db.isnull().sum().sum()}')
df_db.describe().round(3)

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(7, 4))
df_db['Class'].value_counts().plot.bar(ax=ax, color=sns.color_palette('Set2', 7))
ax.set_title('Dry Bean — Class Distribution')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_dry_bean_classes.png', dpi=150)
plt.show()

In [ ]:
# Feature distributions
features = [c for c in df_db.columns if c != 'Class']
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for i, col in enumerate(features):
    ax = axes[i // 4, i % 4]
    ax.hist(df_db[col], bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_title(col, fontsize=10)
# Hide empty subplot
axes[3, 3].set_visible(False)
fig.suptitle('Dry Bean — Feature Distributions', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_dry_bean_features.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = df_db[features].corr()
sns.heatmap(corr, cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            cbar_kws={'shrink': 0.8})
ax.set_title('Dry Bean — Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_dry_bean_corr.png', dpi=150)
plt.show()

---
## 3. Adult Income

In [ ]:
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]
df_ai = pd.read_csv('../data/adult_income/adult.data', header=None, names=columns, 
                     na_values=' ?', skipinitialspace=True)

print(f'Shape: {df_ai.shape}')
print(f'Classes: {df_ai["income"].value_counts().to_dict()}')
print(f'Missing values:\n{df_ai.isnull().sum()[df_ai.isnull().sum() > 0]}')

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(5, 3))
df_ai['income'].value_counts().plot.bar(ax=ax, color=['#3498db', '#e74c3c'])
ax.set_title('Adult Income — Class Distribution')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_adult_income_classes.png', dpi=150)
plt.show()

In [ ]:
# Numerical feature distributions
num_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3, i % 3]
    for inc, color in zip(['<=50K', '>50K'], ['#3498db', '#e74c3c']):
        ax.hist(df_ai[df_ai['income'] == inc][col].dropna(), bins=40, 
                alpha=0.6, color=color, label=inc)
    ax.set_title(col)
    ax.legend(fontsize=8)
fig.suptitle('Adult Income — Numerical Feature Distributions', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_adult_income_num_features.png', dpi=150)
plt.show()

In [ ]:
# Categorical features
cat_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex']
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
for i, col in enumerate(cat_cols):
    ax = axes[i // 4, i % 4]
    df_ai[col].value_counts().plot.bar(ax=ax, color=sns.color_palette('Set2'))
    ax.set_title(col, fontsize=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)
axes[1, 3].set_visible(False)
fig.suptitle('Adult Income — Categorical Feature Distributions', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_adult_income_cat_features.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap (numerical features only)
fig, ax = plt.subplots(figsize=(8, 6))
corr = df_ai[num_cols].corr()
sns.heatmap(corr, cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5,
            annot=True, fmt='.2f',
            cbar_kws={'shrink': 0.8})
ax.set_title('Adult Income — Numerical Feature Correlation')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_adult_income_corr.png', dpi=150)
plt.show()

---
## Summary Comparison

In [ ]:
summary = pd.DataFrame({
    'Dataset': ['Breast Cancer', 'Dry Bean', 'Adult Income'],
    'Samples': [569, 13611, 48842],
    'Features': [30, 16, '108 (after one-hot)'],
    'Classes': [2, 7, 2],
    'Class Balance': [
        '357 benign / 212 malignant',
        'Imbalanced (3546 DERMASON vs 522 BOMBAY)',
        '37155 <=50K / 11687 >50K'
    ],
    'Missing Values': ['None', 'None', 'None (dropped)'],
    'Feature Types': ['All numerical', 'All numerical', 'Mixed (6 categorical + 8 numerical)']
})
summary